In [1]:
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from string import punctuation
import pymorphy3
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from keras.models import Model
from tensorflow.keras.models import Sequential
from keras.layers import Input, Dense, Embedding, SpatialDropout1D, concatenate
from keras.layers import GRU, Bidirectional, GlobalAveragePooling1D, GlobalMaxPooling1D
from keras.preprocessing import text, sequence
from keras.callbacks import Callback
import tensorflow as tf
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
import seaborn as sns
import matplotlib.pyplot as plt

In [1]:
conda list tensorflow

# packages in environment at C:\Users\Safjay\anaconda3:
#
# Name                    Version                   Build  Channel
tensorflow                2.8.0                    pypi_0    pypi
tensorflow-io-gcs-filesystem 0.31.0                   pypi_0    pypi

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Lemmatizzation + preprocessed
df = pd.read_csv('lemmatization+preprocessed.csv')
df['comment'] = df['comment'].astype(str)
df = df.drop('Unnamed: 0', axis=1)
df

,comment,normal,insult,threat,obscenity
0,скотина сказать,0,1,0,0
1,сегодня проезжала рабочей домами снитенко гомо...,1,0,0,0
2,очередной лохотрон придумывать очередной налог...,1,0,0,0
3,ретро дежавю сложно понять чужое сердце лиш ощ...,1,0,0,0
4,статус агрогородка получили,1,0,0,0
...,...,...,...,...,...
248285,правильно пять,1,0,0,0
248286,ебанные нубы заходите сервер ник creepro пвп п...,0,1,0,0
248287,наверное рекорд году училище коренной зуб возм...,1,0,0,0
248288,спасибо всем большое,1,0,0,0


In [3]:
df.head()

,comment,normal,insult,threat,obscenity
0,скотина сказать,0,1,0,0
1,сегодня проезжала рабочей домами снитенко гомо...,1,0,0,0
2,очередной лохотрон придумывать очередной налог...,1,0,0,0
3,ретро дежавю сложно понять чужое сердце лиш ощ...,1,0,0,0
4,статус агрогородка получили,1,0,0,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248290 entries, 0 to 248289
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   comment    248290 non-null  object
 1   normal     248290 non-null  int64 
 2   insult     248290 non-null  int64 
 3   threat     248290 non-null  int64 
 4   obscenity  248290 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 9.5+ MB


In [5]:
df = df.dropna()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248290 entries, 0 to 248289
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   comment    248290 non-null  object
 1   normal     248290 non-null  int64 
 2   insult     248290 non-null  int64 
 3   threat     248290 non-null  int64 
 4   obscenity  248290 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 9.5+ MB


In [7]:
# Разделение на обучающую и тестовую выборки
df_train, df_test = train_test_split(df, train_size=0.8)

In [8]:
def preprocess_text(text):
    text = text.replace("ё", "е")
    text = re.sub('((www\.[^\s]+)|(https?://[^\s]+))', 'URL', text)
    text = re.sub('[^a-zA-Zа-яА-Я]+', ' ', text)
    text = re.sub(' +', ' ', text)
    text = text.lower()
    text = "".join(text)
    return text.strip()

In [9]:
df_train['comment'] = df_train['comment'].apply(preprocess_text).apply(lambda x: preprocess_text(x))
df_test['comment'] = df_test['comment'].apply(preprocess_text).apply(lambda x: preprocess_text(x))

In [10]:
# Определение признаков и целевых переменных
feature_train = df_train['comment'].values
feature_test = df_test['comment'].values
target_train = df_train.drop('comment', axis=1).values
target_test = df_test.drop('comment', axis=1).values

In [11]:
# Токенизация и паддинг
max_features = 30000
maxlen = 300
tokenizer = text.Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(list(feature_train) + list(feature_test))
token_feature_train = tokenizer.texts_to_sequences(feature_train)
token_feature_test = tokenizer.texts_to_sequences(feature_test)
feature_train = sequence.pad_sequences(token_feature_train, maxlen=maxlen)
feature_test = sequence.pad_sequences(token_feature_test, maxlen=maxlen)

In [12]:
# Загрузка эмбеддингов
EMBEDDING_FILE = 'cc.ru.300.vec'
embed_size = 300

In [13]:
def get_coefs(word, *arr):
    return word, np.asarray(arr, dtype='float32')

embeddings_index = dict(get_coefs(*o.rstrip().rsplit(' ')) for o in open(EMBEDDING_FILE, encoding='utf-8'))

In [14]:
# Создание эмбеддинг матрицы
word_index = tokenizer.word_index
nb_words = min(max_features, len(word_index))
embedding_matrix = np.zeros((nb_words, embed_size))
for word, i in word_index.items():
    if i >= max_features:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [15]:
# Финальная модель с оптимальными параметрами
top_batch_size = 2048
top_epochs = 4
top_learning_rate = 0.01
top_dropout_rate = 0.4

In [16]:
def model_final(learning_rate, dropout_rate):
    inp = Input(shape=(maxlen,))
    x = Embedding(max_features, embed_size, weights=[embedding_matrix])(inp)
    x = SpatialDropout1D(dropout_rate)(x)
    x = Bidirectional(GRU(80, return_sequences=True))(x)
    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)
    conc = concatenate([avg_pool, max_pool])
    outp = Dense(4, activation="sigmoid")(conc)
    model = Model(inputs=inp, outputs=outp)

    model.compile(loss='binary_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=['accuracy', 'AUC', tf.keras.metrics.Recall(), tf.keras.metrics.Precision()])
    return model

In [17]:
def model_run(X, y, batch_size, n_epochs, learning_rate, dropout_rate):
    model = model_final(learning_rate, dropout_rate)
    history = model.fit(X, y,
                        batch_size=batch_size,
                        epochs=n_epochs,
                        validation_split=0.1)
    return (history, model)

# Обучение финальной модели
history_2, model_2 = model_run(feature_train, target_train,
                                top_batch_size, top_epochs,
                                top_learning_rate, top_dropout_rate)

Epoch 1/4
88/88 [==============================] - 1526s 17s/step - loss: 0.1375 - accuracy: 0.9156 - auc: 0.9822 - recall: 0.9020 - precision: 0.9132 - val_loss: 0.0745 - val_accuracy: 0.9526 - val_auc: 0.9937 - val_recall: 0.9499 - val_precision: 0.9590
Epoch 2/4
88/88 [==============================] - 1821s 21s/step - loss: 0.0642 - accuracy: 0.9558 - auc: 0.9954 - recall: 0.9545 - precision: 0.9635 - val_loss: 0.0734 - val_accuracy: 0.9506 - val_auc: 0.9935 - val_recall: 0.9496 - val_precision: 0.9590
Epoch 3/4
88/88 [==============================] - 2117s 24s/step - loss: 0.0481 - accuracy: 0.9633 - auc: 0.9972 - recall: 0.9655 - precision: 0.9720 - val_loss: 0.0805 - val_accuracy: 0.9499 - val_auc: 0.9920 - val_recall: 0.9468 - val_precision: 0.9579
Epoch 4/4
88/88 [==============================] - 1912s 22s/step - loss: 0.0357 - accuracy: 0.9695 - auc: 0.9983 - recall: 0.9749 - precision: 0.9794 - val_loss: 0.0956 - val_accuracy: 0.9441 - val_auc: 0.9895 - val_recall: 0.9425 

In [36]:
y_pred = model_2.predict(feature_test, verbose=1, batch_size=2048)

25/25 [==============================] - 74s 3s/step


In [40]:
# Токенизация
max_features = 30000
maxlen = 300

# Используем исходные текстовые данные, а не уже обработанные массивы
# feature_train и feature_test уже содержат паддинг, поэтому они не подходят
# Нужно использовать исходные тексты из df_train и df_test

# Получаем исходные тексты
train_texts = df_train['comment'].values
test_texts = df_test['comment'].values

tokenizer = text.Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(list(train_texts) + list(test_texts))

# Сохраняем токенизатор
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Токенизируем тексты
token_feature_train = tokenizer.texts_to_sequences(train_texts)
token_feature_test = tokenizer.texts_to_sequences(test_texts)

# Паддинг последовательностей
feature_train_pad = sequence.pad_sequences(token_feature_train, maxlen=maxlen)
feature_test_pad = sequence.pad_sequences(token_feature_test, maxlen=maxlen)

In [ ]:
import json

# Сохраняем модель Keras в формате .h5
model_2.save('toxicity_model.h5')
print("Модель сохранена в формате .h5")

model_params = {
    'max_features': max_features,
    'maxlen': maxlen,
    'embed_size': embed_size,
    'learning_rate': top_learning_rate,
    'dropout_rate': top_dropout_rate,
    'batch_size': top_batch_size,
    'epochs': top_epochs
}

with open('model_params.pickle', 'wb') as handle:
    pickle.dump(model_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

target_columns = list(df.drop('comment', axis=1).columns)
with open('target_columns.pickle', 'wb') as handle:
    pickle.dump(target_columns, handle, protocol=pickle.HIGHEST_PROTOCOL)

Модель сохранена в формате .h5
